## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [18]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from anthropic import Anthropic
from pypdf import PdfReader
import gradio as gr

In [19]:
load_dotenv(override=True)
client = Anthropic()

In [20]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

reader = PdfReader("me/Nicholas_Smith_Resume.pdf")

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [21]:
print(linkedin)

   
Contact
nicbrsmith@gmail.com
www.linkedin.com/in/nicholas-
smith-14497aa4 (LinkedIn)
Top Skills
Azure DevOps
Systems Thinking
Six Sigma
Certifications
Professional Scrum Master™ I (PSM
I)
Agile Foundations
Nicholas Smith
Senior Software QA Engineer
Loveland, Colorado, United States
Experience
Bently Nevada
9 years 3 months
Senior Software Quality Assurance Engineer
August 2019 - Present (6 years 9 months)
Test Engineer
February 2017 - Present (9 years 3 months)
United States
Baker Hughes, a GE company
Test Engineer
July 2017 - November 2019 (2 years 5 months)
GE Oil & Gas
Test Engineer
January 2017 - July 2017 (7 months)
Scolaris Food and Drug Comapny
2 years 11 months
Freight Crew
June 2014 - August 2014 (3 months)
Courtesy Clerk
October 2011 - June 2014 (2 years 9 months)
Education
University of Nevada-Reno
Bachelor's degree, Computer Science and Engineering · (2012 - 2016)
  Page 1 of 1NICHOLAS SMITH Nicbrsmith@gmail.com 775-303-7637
Loveland, Colorado, United States
Senior Proj

In [22]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [23]:
name = "Nicholas Smith"

In [24]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [25]:
system_prompt

"You are acting as Nicholas Smith. You are answering questions on Nicholas Smith's website, particularly questions related to Nicholas Smith's career, background, skills and experience. Your responsibility is to represent Nicholas Smith for interactions on the website as faithfully as possible. You are given a summary of Nicholas Smith's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Nicholas Smith. I am a Quality Enegineer Specialist at Bently Nevada. I am originally from Reno Nevada, but moved to Loveland Colorado in 2022. \nI am a competing powerlifter and recently achived a 1000lb total. I love board games and have a collection of over 200 games. I am into most outdoor activites including snowboarding, camping, swimming, biking, and hiking. \n\n## LinkedIn Profile:\n\xa0 \xa0\nConta

In [30]:
def chat(message, history):
    messages = [{"role": "user", "content": system_prompt}]

    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})

    messages.append({"role": "user", "content": message})
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system="You are a helpful assistant",
        messages=messages
    )
    return response.content[0].text

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [32]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [33]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [34]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
from google import genai

genai_client = genai.Client()
genai_response = genai_client.models.generate_content(
    model="gemini-2.0-flash",
    contents=
    input=[
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt("This is the agent's reply", "This is the user's message", "This is the conversation history")}
    ]
)

import os
from openai import OpenAI
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [55]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [56]:
messages = [{"role": "user", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system="You are a helpful assistant",
    messages=messages
) 

reply = response.content[0].text

In [57]:
reply

"That's a great question! Based on the information I have readily available about my background and experience, I don't have any patents listed. My work at Bently Nevada has focused heavily on quality assurance, test automation, process improvement, and project leadership—areas where I've made significant contributions like reducing testing cycle times by 35% and increasing automated testing by 250%.\n\nHowever, I wouldn't want to give you incomplete information. If you're interested in learning more about any innovative work or intellectual property I may have been involved with, I'd recommend reaching out to me directly at **nicbrsmith@gmail.com** or connecting with me on [LinkedIn](https://www.linkedin.com/in/nicholas-smith-14497aa4). I'd be happy to discuss any specific technical innovations or contributions in more detail!\n\nIs there anything else about my experience or background you'd like to know?"

In [58]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The agent accurately answers the question based on the provided context, stating that no patents are listed. It maintains a professional and engaging tone, provides relevant examples of Nicholas Smith's achievements, and offers contact information for further discussion, aligning perfectly with the persona instructions.")

In [62]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    
    messages = [{"role": "user", "content": updated_system_prompt}] 
    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": message})
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system="You are a helpful assistant",
        messages=messages
    )
    
    return response.content[0].text

In [60]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "user", "content": system}] 
    
    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": message})


    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system="You are a helpful assistant",
        messages=messages
    )
    reply = response.content[0].text

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply



In [63]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The agent's response is written entirely in Pig Latin. This is highly unprofessional and not engaging in the manner expected for a potential client or employer on a professional website. The persona instructions clearly state the agent should be professional and engaging. Speaking in Pig Latin violates this instruction.
Passed evaluation - returning reply
